# 01 — Carga del dataset en SQL

Carga los 9 CSV de Olist en una base **DuckDB** (`data/olist.db`) y valida el resultado con
5 consultas. Cada consulta es la semilla de un sprint posterior, no un ejercicio suelto.

| # | Consulta | Alimenta |
|---|---|---|
| 1 | Volumen y ventana temporal | S1 — EDA |
| 2 | Facturación y ticket por categoría | S1 — EDA |
| 3 | Cumplimiento logístico por estado | S1 / S4 |
| 4 | Reseñas vs. retraso de entrega | S3 — insatisfacción |
| 5 | Base RFM por cliente | S2 — segmentación |

> Los CSV no están versionados. Bajarlos de
> [Kaggle — Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
> y descomprimirlos en `data/raw/`.

In [1]:
import duckdb, pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
DB  = Path("../data/olist.db")

assert RAW.exists(), f"No existe {RAW.resolve()} — bajar el dataset de Kaggle primero"
con = duckdb.connect(DB)
pd.set_option("display.width", 120)

## 1. Carga de las 9 tablas

DuckDB lee el CSV directo: no hace falta pasar por pandas ni declarar tipos.

In [2]:
# olist_orders_dataset.csv -> orders ; product_category_name_translation.csv -> product_category_translation
def nombre_tabla(p):
    t = p.stem.removeprefix("olist_").removesuffix("_dataset")
    return "product_category_translation" if t.startswith("product_category") else t

csvs = sorted(RAW.glob("*.csv"))
assert len(csvs) == 9, f"Se esperaban 9 CSV en data/raw/, hay {len(csvs)}"

for p in csvs:
    t = nombre_tabla(p)
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_csv_auto('{p.as_posix()}')")

resumen = con.execute("""
    SELECT table_name AS tabla, estimated_size AS filas, column_count AS columnas
    FROM duckdb_tables() ORDER BY filas DESC
""").df()
resumen

,tabla,filas,columnas
0,geolocation,1000163,5
1,order_items,112650,7
2,order_payments,103886,5
3,customers,99441,5
4,orders,99441,8
5,order_reviews,99224,7
6,products,32951,9
7,sellers,3095,4
8,product_category_translation,71,2


In [3]:
# Chequeo mínimo: si esto falla, la carga se rompió y no tiene sentido seguir
assert len(resumen) == 9, "Faltan tablas"
assert con.execute("SELECT count(*) FROM orders").fetchone()[0] > 99_000, "orders vino corta"
assert con.execute("SELECT count(DISTINCT order_id) FROM orders").fetchone()[0] == con.execute("SELECT count(*) FROM orders").fetchone()[0], "order_id no es único"
print("Carga OK")

Carga OK


## 2. Consulta 1 — Volumen y ventana temporal

Cuánto abarca el dataset y en qué estado quedaron las órdenes. Define qué período es usable:
los meses de los extremos suelen estar incompletos y hay que recortarlos en el EDA.

In [4]:
con.execute("""
    SELECT order_status                              AS estado,
           count(*)                                  AS ordenes,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct,
           min(order_purchase_timestamp)::DATE       AS desde,
           max(order_purchase_timestamp)::DATE       AS hasta
    FROM orders
    GROUP BY estado
    ORDER BY ordenes DESC
""").df()

,estado,ordenes,pct,desde,hasta
0,delivered,96478,97.02,2016-09-15,2018-08-29
1,shipped,1107,1.11,2016-09-04,2018-09-03
2,canceled,625,0.63,2016-09-05,2018-10-17
3,unavailable,609,0.61,2016-10-05,2018-08-21
4,invoiced,314,0.32,2016-10-04,2018-08-14
5,processing,301,0.30,2016-10-05,2018-07-23
6,created,5,0.01,2017-11-06,2018-02-09
7,approved,2,0.00,2017-02-06,2017-04-25


## 3. Consulta 2 — Facturación y ticket por categoría

Top 15 categorías por facturación (precio + flete). Marca dónde está el negocio y qué
categorías tienen volumen suficiente para modelar después.

In [5]:
con.execute("""
    SELECT coalesce(t.product_category_name_english, p.product_category_name, 'sin_categoria') AS categoria,
           count(DISTINCT i.order_id)              AS ordenes,
           round(sum(i.price), 0)                  AS facturacion,
           round(avg(i.price), 2)                  AS precio_medio,
           round(sum(i.freight_value) / sum(i.price) * 100, 1) AS flete_pct_sobre_precio
    FROM order_items i
    JOIN products p                     ON p.product_id = i.product_id
    LEFT JOIN product_category_translation t ON t.product_category_name = p.product_category_name
    GROUP BY categoria
    ORDER BY facturacion DESC
    LIMIT 15
""").df()

,categoria,ordenes,facturacion,precio_medio,flete_pct_sobre_precio
0,health_beauty,8836,1258681.0,130.16,14.5
1,watches_gifts,5624,1205006.0,201.14,8.3
2,bed_bath_table,9417,1036989.0,93.30,19.7
3,sports_leisure,7720,988049.0,114.34,17.1
4,computers_accessories,6689,911954.0,116.51,16.2
5,furniture_decor,6449,729762.0,87.56,23.7
6,cool_stuff,3632,635291.0,167.36,13.2
7,housewares,5884,632249.0,90.79,23.1
8,auto,3897,592720.0,139.96,15.6
9,garden_tools,3518,485256.0,111.63,20.4


## 4. Consulta 3 — Cumplimiento logístico por estado

Entregas efectivas: días reales vs. la fecha estimada que vio el cliente. El `retraso` negativo
es una entrega adelantada. Sólo estados con más de 500 entregas, para que el promedio signifique algo.

In [6]:
con.execute("""
    SELECT c.customer_state                                   AS estado,
           count(*)                                           AS entregas,
           round(avg(datediff('day', o.order_purchase_timestamp, o.order_delivered_customer_date)), 1) AS dias_entrega,
           round(avg(datediff('day', o.order_estimated_delivery_date, o.order_delivered_customer_date)), 1) AS dias_vs_estimado,
           round(100.0 * avg(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END), 1) AS pct_tarde
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY estado
    HAVING count(*) > 500
    ORDER BY pct_tarde DESC
""").df()

,estado,entregas,dias_entrega,dias_vs_estimado,pct_tarde
0,MA,717,21.5,-9.6,19.7
1,CE,1279,21.2,-10.8,15.3
2,BA,3256,19.3,-10.8,14.0
3,RJ,12350,15.2,-11.8,13.5
4,PA,946,23.7,-14.1,12.4
5,ES,1995,15.7,-10.5,12.2
6,MS,701,15.5,-11.1,11.6
7,PB,517,20.4,-13.3,11.0
8,PE,1593,18.4,-13.3,10.8
9,SC,3546,14.9,-11.5,9.8


## 5. Consulta 4 — Reseñas vs. retraso de entrega

La hipótesis del S3: la insatisfacción se explica sobre todo por la logística, no por el producto.
Acá se ve si el retraso separa las reseñas de 1 estrella de las de 5.

In [7]:
con.execute("""
    SELECT r.review_score                                  AS puntaje,
           count(*)                                        AS reseñas,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           round(avg(datediff('day', o.order_purchase_timestamp, o.order_delivered_customer_date)), 1) AS dias_entrega,
           round(100.0 * avg(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END), 1) AS pct_tarde
    FROM order_reviews r
    JOIN orders o ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY puntaje
    ORDER BY puntaje
""").df()

,puntaje,reseñas,pct,dias_entrega,pct_tarde
0,1,9405,9.8,21.3,37.8
1,2,2941,3.1,16.6,20.6
2,3,7961,8.3,14.2,11.0
3,4,18987,19.7,12.3,5.0
4,5,57059,59.2,10.6,3.0


## 6. Consulta 5 — Base RFM por cliente

Entrada directa del S2. Se agrupa por `customer_unique_id` (la persona), **no** por `customer_id`,
que en este dataset es una clave por orden. La recencia se mide contra la última compra del dataset.

In [8]:
rfm = con.execute("""
    WITH corte AS (SELECT max(order_purchase_timestamp) AS fecha FROM orders)
    SELECT c.customer_unique_id                                        AS cliente,
           datediff('day', max(o.order_purchase_timestamp), (SELECT fecha FROM corte)) AS recencia_dias,
           count(DISTINCT o.order_id)                                  AS frecuencia,
           round(sum(pg.payment_value), 2)                             AS monto
    FROM orders o
    JOIN customers c        ON c.customer_id = o.customer_id
    JOIN order_payments pg  ON pg.order_id   = o.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY cliente
""").df()

print(f"{len(rfm):,} clientes únicos · {(rfm.frecuencia > 1).mean():.1%} compró más de una vez")
rfm.describe().round(2)

94,989 clientes únicos · 3.0% compró más de una vez


,recencia_dias,frecuencia,monto
count,94989.00,94989.00,94989.00
mean,287.34,1.03,165.69
std,152.99,0.21,226.74
min,44.00,1.00,9.59
25%,163.00,1.00,63.10
50%,268.00,1.00,107.90
75%,396.00,1.00,182.94
max,773.00,16.00,13664.08


---

**Estado:** S0 cerrado. La base queda en `data/olist.db` (no versionada, se regenera corriendo este notebook).

Siguiente — **S1 (05-oct):** limpieza y `02_eda.ipynb`, 10 gráficos con su lectura.